In [ ]:
import os, sys, glob, json, subprocess, shutil, textwrap

REPO_DIR = "/content/Automodel"
WORK_DIR = "/content/automodel_demo"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
os.makedirs(WORK_DIR, exist_ok=True)


def sh(cmd, check=True):
    print(f"\n$ {cmd}\n" + "-" * 78)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p.returncode

In [ ]:
print("=" * 78)
print("STEP 0 — Checking GPU runtime")
print("=" * 78)

import torch
assert torch.cuda.is_available(), (
    "No GPU found! In Colab: Runtime -> Change runtime type -> select a GPU."
)
GPU_NAME = torch.cuda.get_device_name(0)
BF16_OK = torch.cuda.is_bf16_supported()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GB | bf16 supported: {BF16_OK}")

print("\n" + "=" * 78)
print("STEP 1 — Installing NeMo AutoModel (takes a few minutes)")
print("=" * 78)

if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 https://github.com/NVIDIA-NeMo/Automodel.git {REPO_DIR}")
sh(f"pip -q install -e {REPO_DIR}")
sh("pip -q install pyyaml peft")

sh('python -c "import nemo_automodel; print(\'NeMo AutoModel version:\', '
   'getattr(nemo_automodel, \'__version__\', \'source\'))"')

In [ ]:
print("\n" + "=" * 78)
print("STEP 2 — Preparing the recipe")
print("=" * 78)

import yaml

candidates = sorted(glob.glob(
    os.path.join(REPO_DIR, "examples", "llm_finetune", "qwen", "*0p6b*peft*.yaml")
)) or sorted(glob.glob(
    os.path.join(REPO_DIR, "examples", "llm_finetune", "**", "*peft*.yaml"),
    recursive=True,
))
assert candidates, "Could not find a PEFT recipe in the cloned repo."
BASE_RECIPE = candidates[0]
print(f"Base recipe: {os.path.relpath(BASE_RECIPE, REPO_DIR)}")

with open(BASE_RECIPE) as f:
    cfg = yaml.safe_load(f)

print("\n--- Original recipe (as shipped) ---")
print(yaml.dump(cfg, sort_keys=False)[:2500])


def patch(node):
    if isinstance(node, dict):
        for k, v in list(node.items()):
            if isinstance(v, str) and not BF16_OK and v.lower() in (
                    "bf16", "bfloat16", "torch.bfloat16"):
                node[k] = "float32"
            elif k in ("batch_size", "local_batch_size") and isinstance(v, int):
                node[k] = min(v, 4)
            elif k == "global_batch_size" and isinstance(v, int):
                node[k] = min(v, 8)
            else:
                patch(v)
    elif isinstance(node, list):
        for item in node:
            patch(item)


patch(cfg)

cfg.setdefault("step_scheduler", {})
cfg["step_scheduler"]["max_steps"] = 40
cfg["step_scheduler"]["ckpt_every_steps"] = 40
cfg["step_scheduler"]["num_epochs"] = 1

if isinstance(cfg.get("checkpoint"), dict):
    cfg["checkpoint"]["enabled"] = True
    cfg["checkpoint"]["checkpoint_dir"] = CKPT_DIR

DEMO_RECIPE = os.path.join(WORK_DIR, "qwen3_0p6b_colab_lora.yaml")
with open(DEMO_RECIPE, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("\n--- Patched recipe (what we will actually run) ---")
print(yaml.dump(cfg, sort_keys=False)[:2500])

MODEL_ID = "Qwen/Qwen3-0.6B"
try:
    MODEL_ID = cfg["model"]["pretrained_model_name_or_path"]
except Exception:
    pass
print(f"\nBase model: {MODEL_ID}")

In [ ]:
print("\n" + "=" * 78)
print("STEP 3 — Training (LoRA fine-tune of Qwen3-0.6B on HellaSwag)")
print("=" * 78)

env_prefix = "HF_HUB_ENABLE_HF_TRANSFER=0 TOKENIZERS_PARALLELISM=false"
rc = sh(f"cd {WORK_DIR} && {env_prefix} automodel {DEMO_RECIPE}", check=False)
if rc != 0:
    print("\nRetrying with legacy CLI syntax...")
    sh(f"cd {WORK_DIR} && {env_prefix} automodel finetune llm -c {DEMO_RECIPE}")

In [ ]:
print("\n" + "=" * 78)
print("STEP 4 — Evaluating: base model vs LoRA fine-tuned model")
print("=" * 78)

from transformers import AutoModelForCausalLM, AutoTokenizer

DTYPE = torch.bfloat16 if BF16_OK else torch.float32
PROMPT = ("A man is sitting on a roof. He starts pulling up roofing shingles. "
          "What happens next?")


def generate(model, tok, prompt, max_new_tokens=60):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=None, top_p=None,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:],
                      skip_special_tokens=True)


tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map="cuda")

print("\n[BASE MODEL]")
print(textwrap.fill(generate(base, tok, PROMPT), 90))

ckpt_glob = sorted(glob.glob(os.path.join(CKPT_DIR, "**", "model"),
                             recursive=True))
if not ckpt_glob:
    ckpt_glob = sorted(glob.glob(os.path.join(WORK_DIR, "**",
                                              "adapter_model.safetensors"),
                                 recursive=True))
    ckpt_glob = [os.path.dirname(p) for p in ckpt_glob]

if ckpt_glob:
    ADAPTER_DIR = ckpt_glob[-1]
    print(f"\nFound checkpoint: {ADAPTER_DIR}")
    try:
        from peft import PeftModel
        tuned = PeftModel.from_pretrained(base, ADAPTER_DIR)
        print("\n[FINE-TUNED MODEL (base + LoRA adapter)]")
        print(textwrap.fill(generate(tuned, tok, PROMPT), 90))
    except Exception as e:
        print(f"\nCould not auto-load the adapter with peft ({e}).")
        print("Inspect the checkpoint contents manually:")
        for p in glob.glob(os.path.join(ADAPTER_DIR, "*"))[:20]:
            print("  ", p)
else:
    print("\nNo checkpoint found — check the training logs above.")

del base
torch.cuda.empty_cache()

In [1]:
print("\n" + "=" * 78)
print("STEP 5 — Bonus: drop-in Python API")
print("=" * 78)

try:
    from nemo_automodel import NeMoAutoModelForCausalLM
    nm = NeMoAutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE).to("cuda")
    print("[NeMoAutoModelForCausalLM]")
    print(textwrap.fill(generate(nm, tok, "The key idea of LoRA is"), 90))
    del nm
    torch.cuda.empty_cache()
except Exception as e:
    print(f"Python-API demo skipped on this version/GPU: {e}")

print("\n" + "=" * 78)
print("DONE! Where to go next:")
print("=" * 78)
print(f"""
1. Recipes to explore (in {REPO_DIR}/examples/):
     llm_finetune/   — SFT + LoRA for Llama, Qwen, Gemma, Phi, GPT-OSS, ...
     llm_pretrain/   — e.g. nanoGPT on FineWeb, DeepSeek-V3 pre-training
     vlm_finetune/   — Qwen-VL, Gemma-3-VL, and other vision-language models
     diffusion/      — FLUX / Wan / Qwen-Image LoRA fine-tuning
2. Swap the model: override one field —
     automodel recipe.yaml --model.pretrained_model_name_or_path <any-HF-LM>
3. Scale out: the SAME recipe runs on 8 GPUs with `--nproc-per-node 8`,
   or multi-node via the shipped slurm.sub / SkyPilot / Kubernetes launchers.
4. Docs: https://docs.nvidia.com/nemo/automodel/latest/index.html
""")

STEP 0 — Checking GPU runtime
GPU: Tesla T4 | VRAM: 15.6 GB | bf16 supported: True

STEP 1 — Installing NeMo AutoModel (takes a few minutes)

$ git clone --depth 1 https://github.com/NVIDIA-NeMo/Automodel.git /content/Automodel
------------------------------------------------------------------------------
Cloning into '/content/Automodel'...

$ pip -q install -e /content/Automodel
------------------------------------------------------------------------------
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 80

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


[BASE MODEL]
 A. He will pull the shingles up. B. He will pull the shingles down. C. He will pull the
shingles back down. D. He will pull the shingles back up. Answer: The man is sitting on a
roof and starts pulling up roofing shingles. The

Found checkpoint: /content/automodel_demo/checkpoints/epoch_0_step_39/model

Could not auto-load the adapter with peft (Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported).
Inspect the checkpoint contents manually:
   /content/automodel_demo/checkpoints/epoch_0_step_39/model/vocab.json
   /content/automodel_demo/checkpoints/epoch_0_step_39/model/adapter_model.safetensors
   /content/automodel_demo/checkpoints/epoch_0_step_39/model/chat_template.jinja
   /content/automodel_demo/checkpoints/epoch_0_step_39/model/tokenizer_config.json
   /content/automodel_demo/checkpoints/epoch_0_step_39/model/automodel_peft_config.json
   /content/automodel_demo/checkpoints/epoch_0_step_39/model/adapter_conf